# Data Check: Order Book Derived

Quick visual check for pipeline data quality.

1. Run the order_book_derived pipeline steps to populate parquet data
2. Load each data type and plot for visual inspection

## Setup

In [ ]:
import sys

sys.path.insert(0, "../../src")

import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from qate.core.model import ExchangeName, Market
from qate.core.symbol import Symbol
from qate.util.dt_range import DtRange

from makalu import settings as sys_env
from makalu.storage.parquet_store import ParquetStore
from pipeline.market_pair import EXECUTION_SIZE

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
%matplotlib inline

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s"
)

POPULATE_DATA = False

MARKET = Market(ExchangeName.GMO, Symbol.BTC_JPY)
DATA_START_DATE = "20260112"
DATA_END_DATE = "20260113"

DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)
EXECUTION_SIZE_STR = EXECUTION_SIZE.get(MARKET.symbol)

print(f"Market:         {MARKET.id}")
print(f"Date range:     {DATA_START_DATE} - {DATA_END_DATE}")
print(f"Execution size: {EXECUTION_SIZE_STR}")

## 1. Run Pipeline Steps

Populate the parquet store by running the OrderBookDerived stage with two steps
(MarketPrice is derived by Hase: `hase derive MarketPrice --market ... --execution-size ...`):
- **PopulateBookStateStep** (order book state metrics)
- **PopulateBookChangeStep** (order book change metrics)

In [ ]:
from pipeline.framework import Pipeline
from pipeline.order_book_derived_steps import (
    PopulateBookChangeStep,
    PopulateBookStateStep,
)
from pipeline.stages import OrderBookDerivedStage


def populate_data():
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

    pipeline = Pipeline(parquet_store)
    pipeline.stage(
        OrderBookDerivedStage(MARKET)
        .step(
            PopulateBookStateStep(
                volume_imbalance_k=[1, 3, 5, 10],
                weighted_imbalance_k=[1, 3, 5, 10],
                depth_imbalance_rate=[0.0001, 0.0002, 0.0005, 0.001],
                microprice_k=[1, 3, 5, 10],
            )
        )
        .step(PopulateBookChangeStep(k_levels=[1, 3, 5, 10]))
    )

    print("Running pipeline...")
    pipeline.process(DT_RANGE)
    print("Done.")


if POPULATE_DATA:
    populate_data()

## 2. Execution Price (MarketPrice)

In [ ]:
store = ParquetStore(sys_env.get_parquet_root_dir())
ep_df = store.create_execution_price(MARKET, EXECUTION_SIZE_STR).read(DT_RANGE)

print(f"Shape: {ep_df.shape}")
print(f"Time range: {ep_df.index.min()} ~ {ep_df.index.max()}")
print(f"NaN counts:\n{ep_df.isna().sum()}")
display(ep_df.describe())
display(ep_df.head())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(ep_df.index, ep_df["mid"], linewidth=0.5)
axes[0].set_ylabel("Mid Price")
axes[0].set_title(f"{MARKET.id} Execution Price")

axes[1].plot(ep_df.index, ep_df["spread"], linewidth=0.5, color="orange")
axes[1].set_ylabel("Spread")

axes[2].plot(ep_df.index, ep_df["spread_bps"], linewidth=0.5, color="red")
axes[2].set_ylabel("Spread (bps)")
axes[2].set_xlabel("Time")

plt.tight_layout()
plt.show()

## 3. Book State

In [ ]:
bs_df = store.create_single_market("BookState", MARKET).read(DT_RANGE)

print(f"Shape: {bs_df.shape}")
print(f"Time range: {bs_df.index.min()} ~ {bs_df.index.max()}")
print(f"Columns: {list(bs_df.columns)}")
print(f"NaN counts:\n{bs_df.isna().sum()}")
display(bs_df.describe())
display(bs_df.head())

In [ ]:
# Volume imbalance
vol_imb_cols = [c for c in bs_df.columns if c.startswith("vol_imb_")]
fig, axes = plt.subplots(
    len(vol_imb_cols), 1, figsize=(15, 3 * len(vol_imb_cols)), sharex=True
)
for ax, col in zip(axes, vol_imb_cols):
    ax.plot(bs_df.index, bs_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Volume Imbalance")
plt.tight_layout()
plt.show()

In [ ]:
# Weighted imbalance
weighted_cols = [c for c in bs_df.columns if c.startswith("weighted_imb_")]
fig, axes = plt.subplots(
    len(weighted_cols), 1, figsize=(15, 3 * len(weighted_cols)), sharex=True
)
for ax, col in zip(axes, weighted_cols):
    ax.plot(bs_df.index, bs_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Weighted Imbalance")
plt.tight_layout()
plt.show()

In [ ]:
# Depth imbalance
depth_cols = [c for c in bs_df.columns if c.startswith("depth_imb_")]
fig, axes = plt.subplots(
    len(depth_cols), 1, figsize=(15, 3 * len(depth_cols)), sharex=True
)
for ax, col in zip(axes, depth_cols):
    ax.plot(bs_df.index, bs_df[col], linewidth=0.3, alpha=0.7)
    ax.set_ylabel(col)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"{MARKET.id} Depth Imbalance")
plt.tight_layout()
plt.show()

In [ ]:
# Microprice
mp_cols = [c for c in bs_df.columns if c.startswith("microprice_")]
fig, axes = plt.subplots(
    len(mp_cols) + 1, 1, figsize=(15, 3 * (len(mp_cols) + 1)), sharex=True
)
axes[0].plot(bs_df.index, bs_df["best_mid"], linewidth=0.5)
axes[0].set_ylabel("best_mid")
axes[0].set_title(f"{MARKET.id} Microprice vs Mid")
for ax, col in zip(axes[1:], mp_cols):
    ax.plot(bs_df.index, bs_df[col], linewidth=0.3, alpha=0.7, color="orange")
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

## 4. Book Change

In [ ]:
bc_df = store.create_single_market("BookChange", MARKET).read(DT_RANGE)

print(f"Shape: {bc_df.shape}")
print(f"Time range: {bc_df.index.min()} ~ {bc_df.index.max()}")
print(f"Columns: {list(bc_df.columns)}")
print(f"NaN counts:\n{bc_df.isna().sum()}")
display(bc_df.describe())
display(bc_df.head())

In [ ]:
# Delta sec (time between snapshots)
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(bc_df.index, bc_df["delta_sec"], linewidth=0.3, alpha=0.7)
ax.set_ylabel("delta_sec")
ax.set_title(f"{MARKET.id} Snapshot Delta (sec)")
plt.tight_layout()
plt.show()

print("delta_sec stats:")
display(bc_df["delta_sec"].describe())

In [ ]:
# Depth change rates per k-level
k_levels = sorted(set(int(c.split("_k")[1]) for c in bc_df.columns if "_k" in c))

for k in k_levels:
    cols = [c for c in bc_df.columns if c.endswith(f"_k{k}")]
    fig, axes = plt.subplots(len(cols), 1, figsize=(15, 2.5 * len(cols)), sharex=True)
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        ax.plot(bc_df.index, bc_df[col], linewidth=0.3, alpha=0.7)
        ax.set_ylabel(col, fontsize=8)
        ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
    axes[0].set_title(f"{MARKET.id} Book Change (k={k})")
    plt.tight_layout()
    plt.show()

## 5. Data Consistency Check

In [ ]:
print("=" * 60)
print("Data Consistency Summary")
print("=" * 60)

for name, df in [
    ("ExecutionPrice", ep_df),
    ("BookState", bs_df),
    ("BookChange", bc_df),
]:
    print(f"\n--- {name} ---")
    print(f"  Rows:       {len(df):,}")
    print(f"  Columns:    {len(df.columns)}")
    print(f"  Time range: {df.index.min()} ~ {df.index.max()}")
    total_nan = df.isna().sum().sum()
    print(f"  Total NaN:  {total_nan:,} ({total_nan / df.size * 100:.2f}%)")
    dupes = df.index.duplicated().sum()
    print(f"  Duplicate timestamps: {dupes}")
    monotonic = df.index.is_monotonic_increasing
    print(f"  Monotonic index: {monotonic}")

# Sanity checks on execution price
print("\n--- Execution Price Sanity ---")
bid_leq_ask = (ep_df["bid"] <= ep_df["ask"]).all()
print(f"  bid <= ask: {bid_leq_ask}")
positive_spread = (ep_df["spread"] >= 0).all()
print(f"  spread >= 0: {positive_spread}")

# Row count comparison
print("\n--- Row Count Comparison ---")
print(f"  ExecutionPrice: {len(ep_df):,}")
print(f"  BookState:      {len(bs_df):,}")
print(f"  BookChange:     {len(bc_df):,}")